# Etapa 1 — EDA + Modelos Baseline

**Problema de negócio:** Uma operadora de telecomunicações precisa identificar clientes com risco de cancelamento (churn) para agir proativamente.

**Dataset:** Telco Customer Churn — IBM (Kaggle) — 7.043 clientes, 33 variáveis.

**Objetivo:** Explorar os dados, entender distribuições e treinar modelos baseline rastreados no MLflow.

## 0. Setup

In [1]:


import logging
import warnings

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score

from customer_churn_ibm.config import (
    CV_FOLDS,
    MLFLOW_EXPERIMENT,
    NUMERICAL_FEATURES,
    SEED,
    TARGET,
)
from customer_churn_ibm.data import build_preprocessor, clean_data, get_splits, load_raw_data
from customer_churn_ibm.model_baseline import BASELINE_CONFIGS, train_baselines

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(SEED)

print(f'SEED={SEED} | CV_FOLDS={CV_FOLDS}')

ModuleNotFoundError: No module named 'scipy'

## 1. Carregamento dos Dados

In [ ]:
df_raw = load_raw_data()
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

## 2. Qualidade dos Dados

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({'missing': missing, 'missing_%': missing_pct}).query('missing > 0')

In [ ]:
print(f'Duplicatas: {df_raw.duplicated().sum()}')
print(f'Total Charges (antes conversão): {df_raw["Total Charges"].dtype}')
# Total Charges é string — deve ser convertida
n_non_numeric = pd.to_numeric(df_raw['Total Charges'], errors='coerce').isna().sum()
print(f'Valores não-numéricos em Total Charges: {n_non_numeric}')

## 3. Distribuição do Target

In [ ]:
churn_counts = df_raw[TARGET].value_counts()
churn_pct = (churn_counts / len(df_raw) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
churn_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Contagem por Classe')
axes[0].set_xlabel('Churn Value (0=Não, 1=Sim)')
axes[0].set_ylabel('Clientes')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(churn_counts, labels=['Não Churn', 'Churn'], autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Proporção de Churn')
plt.tight_layout()
plt.show()

print(churn_pct.to_string())
print(f'\nDesbalanceamento: {churn_pct[0]/churn_pct[1]:.1f}x mais não-churn que churn')

## 4. Features Numéricas

In [ ]:
df_clean = clean_data(df_raw)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(NUMERICAL_FEATURES):
    ax_hist = axes[0, i]
    ax_box = axes[1, i]
    for label, grp in df_clean.groupby(TARGET):
        ax_hist.hist(grp[col], alpha=0.5, bins=30, label=f'Churn={label}')
    ax_hist.set_title(col)
    ax_hist.legend()
    df_clean.boxplot(column=col, by=TARGET, ax=ax_box)
    ax_box.set_title('')
    ax_box.set_xlabel('Churn Value')

plt.suptitle('Distribuição das Features Numéricas por Classe', y=1.01)
plt.tight_layout()
plt.show()

df_clean.groupby(TARGET)[NUMERICAL_FEATURES].agg(['mean', 'median', 'std']).round(2)

## 5. Features Categóricas — Taxa de Churn

In [ ]:
key_cat = ['Contract', 'Internet Service', 'Payment Method', 'Tech Support',
           'Online Security', 'Senior Citizen']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(key_cat):
    churn_rate = (
        df_clean.groupby(col)[TARGET].mean().sort_values(ascending=False)
    )
    churn_rate.plot(kind='bar', ax=axes[i], color='tomato', alpha=0.8)
    axes[i].set_title(f'Taxa de Churn — {col}')
    axes[i].set_ylabel('Taxa de Churn')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=30)
    for bar in axes[i].patches:
        axes[i].annotate(f'{bar.get_height():.1%}',
                         (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                         ha='center', va='bottom', fontsize=8)

plt.suptitle('Taxa de Churn por Feature Categórica', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Correlações — Features Numéricas

In [ ]:
corr_cols = NUMERICAL_FEATURES + [TARGET]
corr = df_clean[corr_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5)
plt.title('Matriz de Correlação (Numéricas + Target)')
plt.tight_layout()
plt.show()

## 7. Preparação dos Dados

In [ ]:
X_train, X_test, y_train, y_test = get_splits(df_clean)
preprocessor = build_preprocessor()

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Churn rate treino: {y_train.mean():.2%} | Churn rate teste: {y_test.mean():.2%}')

## 8. Baseline com MLflow

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)
results = train_baselines(preprocessor, X_train, X_test, y_train, y_test)
results

## 9. Validação Cruzada Estratificada

In [ ]:
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
cv_rows = []

for name, cfg in BASELINE_CONFIGS.items():
    pipeline = Pipeline([('preprocessor', build_preprocessor()), ('model', cfg['model'])])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_rows.append({'model': name, 'AUC-ROC mean': scores.mean(), 'AUC-ROC std': scores.std()})
    print(f'{name:22s}  AUC-ROC = {scores.mean():.4f} ± {scores.std():.4f}')

pd.DataFrame(cv_rows).sort_values('AUC-ROC mean', ascending=False).reset_index(drop=True)

## 10. Conclusões da Etapa 1

- **Dataset desbalanceado** (~73% não-churn, ~27% churn) — uso de `pos_weight` e métricas além de acurácia.
- **Features mais relevantes (categóricas):** `Contract` (Month-to-month tem taxa de churn ~43%), `Internet Service` (Fiber optic ~41%), `Tech Support` / `Online Security` ausentes.
- **Features numéricas:** `Tenure Months` tem forte correlação negativa com churn — clientes mais antigos tendem a ficar.
- **Leakage evitado:** `Churn Score` e `CLTV` foram removidos por serem derivados do target.
- **Melhor baseline:** GradientBoosting supera Logistic Regression em F1 e AUC-ROC.